In [2]:
# ============================================================
# TASK 2: Quantitative Evaluation
# Novelty Rate & Diversity Score for all three models
# ============================================================
# This notebook loads the three trained models from Task 1,
# generates a batch of names from each, then computes:
#   - Novelty Rate  : % of generated names NOT in training set
#   - Diversity     : unique generated names / total generated
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import random
import pandas as pd
from google.colab import files

# ============================================================
# STEP 0: Upload Required Files
# ============================================================
# You need to upload:
#   1. TrainingNames.txt       (original dataset)
#   2. vanilla_rnn.pt          (saved from Task 1)
#   3. blstm.pt                (saved from Task 1)
#   4. rnn_attention.pt        (saved from Task 1)
# ============================================================

print("Upload TrainingNames.txt and all three .pt model files...")
uploaded = files.upload()

# Load training names into a set for O(1) lookup during novelty check
with open("TrainingNames.txt", "r", encoding="utf-8") as f:
    training_names = set(line.strip().lower() for line in f if line.strip())

print(f"Training set size: {len(training_names)} names")


# ============================================================
# STEP 1: Rebuild Vocabulary (must match Task 1 exactly)
# ============================================================
# The vocabulary must be rebuilt identically to Task 1 so that
# character indices map correctly when loading saved weights.

all_chars     = sorted(set("".join(training_names)))
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab          = special_tokens + all_chars
char2idx       = {ch: idx for idx, ch in enumerate(vocab)}
idx2char       = {idx: ch  for idx, ch in enumerate(vocab)}

PAD_IDX    = char2idx["<PAD>"]
SOS_IDX    = char2idx["<SOS>"]
EOS_IDX    = char2idx["<EOS>"]
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")


# ============================================================
# STEP 2: Rebuild Model Architectures (identical to Task 1)
# ============================================================
# PyTorch's load_state_dict() requires the model class to be
# defined before weights can be loaded into it.

# ---- Vanilla RNN ----------------------------------------
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size   = embedding_dim,
            hidden_size  = hidden_size,
            num_layers   = num_layers,
            batch_first  = True,
            dropout      = dropout if num_layers > 1 else 0,
            nonlinearity = 'tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded       = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded, hidden)
        logits         = self.fc(self.dropout(output))
        return logits, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)


# ---- Bidirectional LSTM ---------------------------------
class BiLSTMEncoderDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(BiLSTMEncoderDecoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.dropout     = nn.Dropout(dropout)
        self.encoder = nn.LSTM(
            input_size    = embedding_dim,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0,
            bidirectional = True
        )
        # Project concatenated fwd+bwd hidden/cell (2H) down to H
        self.h_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.c_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.decoder = nn.LSTM(
            input_size  = embedding_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def _encode_sos(self, device):
        """Encode [SOS] to get warm decoder initial state."""
        sos       = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
        emb       = self.dropout(self.embedding(sos))
        _, (h, c) = self.encoder(emb)
        B         = 1
        h = h.view(self.num_layers, 2, B, self.hidden_size)
        c = c.view(self.num_layers, 2, B, self.hidden_size)
        h = torch.tanh(self.h_proj(
                torch.cat([h[:, 0, :, :], h[:, 1, :, :]], dim=2)))
        c = torch.tanh(self.c_proj(
                torch.cat([c[:, 0, :, :], c[:, 1, :, :]], dim=2)))
        return h, c

    def forward(self, x, hidden=None):
        device      = x.device
        B           = x.size(0)
        h, c        = self._encode_sos(device)
        h           = h.expand(-1, B, -1).contiguous()
        c           = c.expand(-1, B, -1).contiguous()
        emb         = self.dropout(self.embedding(x))
        out, hidden = self.decoder(emb, (h, c))
        return self.fc(self.dropout(out)), hidden

    def init_hidden(self, batch_size, device):
        h = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        c = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        return (h, c)


# ---- Attention Module -----------------------------------
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v    = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len          = encoder_outputs.size(1)
        hidden_expanded  = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy           = torch.tanh(
            self.attn(torch.cat([hidden_expanded, encoder_outputs], dim=2))
        )
        attention_scores = self.v(energy).squeeze(2)
        attn_weights     = torch.softmax(attention_scores, dim=1)
        context          = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context.squeeze(1), attn_weights


# ---- RNN with Attention ---------------------------------
class RNNWithAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(RNNWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size   = embedding_dim,
            hidden_size  = hidden_size,
            num_layers   = num_layers,
            batch_first  = True,
            dropout      = dropout if num_layers > 1 else 0,
            nonlinearity = 'tanh'
        )
        self.attention = Attention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden=None):
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        T      = rnn_out.size(1)
        logits = []
        for t in range(T):
            ctx, _ = self.attention(hidden[-1], rnn_out[:, :t+1, :])
            step   = torch.cat([rnn_out[:, t, :], ctx], dim=1)
            logits.append(self.fc(self.dropout(step)))
        return torch.stack(logits, dim=1), hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)

    def decode_step(self, x, hidden, history):
        """Single inference step — grows history by 1 each call."""
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        history         = torch.cat([history, rnn_out], dim=1)
        ctx, _          = self.attention(hidden[-1], history)
        combined        = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits          = self.fc(self.dropout(combined))
        return logits, hidden, history


# ============================================================
# STEP 3: Load Saved Weights into Models
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Instantiate each architecture with the same hyperparameters as Task 1
rnn_model  = VanillaRNN(VOCAB_SIZE).to(device)
lstm_model = BiLSTMEncoderDecoder(VOCAB_SIZE).to(device)
attn_model = RNNWithAttention(VOCAB_SIZE).to(device)

# Load the saved state dictionaries — map_location handles GPU→CPU moves
rnn_model.load_state_dict(
    torch.load("vanilla_rnn.pt", map_location=device))

lstm_model.load_state_dict(
    torch.load("blstm.pt", map_location=device))

attn_model.load_state_dict(
    torch.load("rnn_attention.pt", map_location=device))

# Switch all models to evaluation mode — disables dropout for inference
rnn_model.eval()
lstm_model.eval()
attn_model.eval()

print("All three models loaded successfully.")


# ============================================================
# STEP 4: Name Generation Function
# ============================================================

def generate_name_rnn(model, device, max_len=20, temperature=0.8):
    """
    Autoregressively generates one name from the model.
    Starts from <SOS>, samples next character at each step,
    stops at <EOS> or max_len characters.

    Temperature controls the softmax sharpness:
      lower → safer/more repetitive names
      higher → more creative/diverse names
    """
    with torch.no_grad():
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        hidden     = model.init_hidden(batch_size=1, device=device)
        generated  = []

        for _ in range(max_len):
            logits, hidden = model(input_char, hidden)

            # Scale logits by temperature before sampling
            probs    = torch.softmax(logits[0, 0] / temperature, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_name_bilstm(model, device, max_len=20, temperature=0.8):
    """
    Autoregressively generates one name from the model.
    Starts from <SOS>, samples next character at each step,
    stops at <EOS> or max_len characters.

    Temperature controls the softmax sharpness:
      lower → safer/more repetitive names
      higher → more creative/diverse names
    """
    with torch.no_grad():
        h, c  = model._encode_sos(device)
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        generated  = []

        for _ in range(max_len):
            emb         = model.dropout(model.embedding(input_char))
            out, (h, c) = model.decoder(emb, (h, c))

            # Scale logits by temperature before sampling
            probs    = torch.softmax(out[0, 0] @ model.fc.weight.T
                                     + model.fc.bias, dim=0)
            probs    = torch.softmax(model.fc(out[0, 0]) / temperature, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_name_attn(model, device, max_len=20, temperature=0.8):
    """
    Autoregressively generates one name from the model.
    Starts from <SOS>, samples next character at each step,
    stops at <EOS> or max_len characters.

    Temperature controls the softmax sharpness:
      lower → safer/more repetitive names
      higher → more creative/diverse names
    """
    with torch.no_grad():
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        hidden     = model.init_hidden(batch_size=1, device=device)
        generated  = []

        # Bootstrap: seed history with the real SOS RNN output
        emb             = model.dropout(model.embedding(input_char))
        rnn_out, hidden = model.rnn(emb, hidden)
        history         = rnn_out  # (1, 1, H)

        # First character prediction from SOS context
        ctx, _   = model.attention(hidden[-1], history)
        combined = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits   = model.fc(model.dropout(combined))
        probs    = torch.softmax(logits[0] / temperature, dim=0)
        next_idx = torch.multinomial(probs, num_samples=1).item()

        if next_idx == EOS_IDX:
            return ""
        if next_idx not in (PAD_IDX, SOS_IDX):
            generated.append(idx2char[next_idx])

        input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

        for _ in range(max_len - 1):
            # Scale logits by temperature before sampling
            logits, hidden, history = model.decode_step(
                input_char, hidden, history)
            probs    = torch.softmax(logits[0] / temperature, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_batch(gen_fn, model, device, n=500, temperature=0.8):
    """
    Generates n names from a model, filtering out:
      - empty strings
      - single-character outputs (not a real name)
    Returns a list of lowercased name strings.
    """
    names, attempts = [], 0
    while len(names) < n and attempts < n * 20:
        name = gen_fn(model, device, max_len=20, temperature=temperature)
        if len(name) > 1:           # discard empty / single-char outputs
            names.append(name.lower())
        attempts += 1
    return names


# ============================================================
# STEP 5: Evaluation Metrics
# ============================================================

def novelty_rate(generated_names, training_set):
    """
    Novelty Rate = (names NOT in training set) / total generated × 100

    A high novelty rate means the model is creative and not
    simply memorising training examples.

    Args:
        generated_names : list of generated name strings
        training_set    : set of training name strings

    Returns:
        float: novelty percentage (0–100)
    """
    novel_count = sum(1 for name in generated_names
                      if name not in training_set)
    return (novel_count / len(generated_names)) * 100


def diversity_score(generated_names):
    """
    Diversity = unique names / total generated names × 100

    A high diversity score means the model produces varied output
    rather than repeating the same names over and over.

    Args:
        generated_names : list of generated name strings

    Returns:
        float: diversity percentage (0–100)
    """
    unique_count = len(set(generated_names))
    return (unique_count / len(generated_names)) * 100


def average_name_length(generated_names):
    """
    Average character length of generated names.
    Useful for checking whether outputs are plausible name lengths.
    """
    return np.mean([len(name) for name in generated_names])


def length_distribution(generated_names):
    """
    Returns a dict mapping name length → count.
    Helps spot if a model is stuck generating only short/long names.
    """
    lengths = [len(n) for n in generated_names]
    dist    = {}
    for l in sorted(set(lengths)):
        dist[l] = lengths.count(l)
    return dist


# ============================================================
# STEP 6: Run Evaluation for All Three Models
# ============================================================

N_GENERATE  = 500    # number of names to generate per model
TEMPERATURE = 0.8    # sampling temperature (same for fair comparison)

print(f"\nGenerating {N_GENERATE} names per model "
      f"(temperature={TEMPERATURE})...\n")

models_info = [
    ("Vanilla RNN",            rnn_model,  generate_name_rnn),
    ("Bidirectional LSTM",     lstm_model, generate_name_bilstm),
    ("RNN + Attention",        attn_model, generate_name_attn),
]

# Store all results for the comparison table
results = []

# Store generated names for detailed inspection
all_generated = {}

for model_name, model, gen_fn in models_info:
    print(f"  Generating from {model_name}...")

    # Generate a batch of names
    generated = generate_batch(gen_fn, model, device,
                                n=N_GENERATE, temperature=TEMPERATURE)
    all_generated[model_name] = generated

    # Compute metrics
    novelty   = novelty_rate(generated, training_names)
    diversity = diversity_score(generated)
    avg_len   = average_name_length(generated)
    unique_n  = len(set(generated))

    results.append({
        "Model"          : model_name,
        "Generated"      : N_GENERATE,
        "Unique Names"   : unique_n,
        "Novelty Rate %"  : round(novelty,   2),
        "Diversity %"     : round(diversity, 2),
        "Avg Name Length" : round(avg_len,   2),
    })

    print(f"    Novelty  : {novelty:.2f}%")
    print(f"    Diversity: {diversity:.2f}%")
    print(f"    Avg Len  : {avg_len:.2f} chars")
    print()


# ============================================================
# STEP 7: Print Formal Comparison Table
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("            QUANTITATIVE EVALUATION REPORT")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

# Save the report as CSV for submission
results_df.to_csv("evaluation_report.csv", index=False)
print("\nReport saved as: evaluation_report.csv")


# ============================================================
# STEP 8: Per-Model Sample Output
# ============================================================
# Shows 15 generated names from each model so you can
# visually inspect the quality alongside the numbers.

print("\n" + "="*70)
print("            SAMPLE GENERATED NAMES (15 per model)")
print("="*70)

for model_name, generated in all_generated.items():
    # Pick 15 unique names from the generated batch
    samples = random.sample(list(set(generated)),
                            min(15, len(set(generated))))
    print(f"\n  {model_name}:")
    # Display in 3 columns of 5 for compact layout
    for i in range(0, len(samples), 3):
        row = samples[i:i+3]
        print("    " + "   ".join(f"{n.capitalize():<15}" for n in row))


# ============================================================
# STEP 9: Length Distribution Analysis
# ============================================================

print("\n" + "="*70)
print("            NAME LENGTH DISTRIBUTION")
print("="*70)

for model_name, generated in all_generated.items():
    dist = length_distribution(generated)
    print(f"\n  {model_name}:")
    print(f"  {'Length':<10} {'Count':<10} {'Bar'}")
    print(f"  {'-'*40}")
    for length, count in sorted(dist.items()):
        bar = "█" * (count // 5)   # 1 block per 5 names
        print(f"  {length:<10} {count:<10} {bar}")


# ============================================================
# STEP 10: Download Results
# ============================================================

files.download("evaluation_report.csv")
print("\nDownload complete.")

Upload TrainingNames.txt and all three .pt model files...


Saving blstm.pt to blstm.pt
Saving rnn_attention.pt to rnn_attention.pt
Saving vanilla_rnn.pt to vanilla_rnn.pt
Training set size: 1000 names
Vocabulary size: 29

Using device: cpu
All three models loaded successfully.

Generating 500 names per model (temperature=0.8)...

  Generating from Vanilla RNN...
    Novelty  : 96.20%
    Diversity: 91.00%
    Avg Len  : 5.79 chars

  Generating from Bidirectional LSTM...
    Novelty  : 38.20%
    Diversity: 83.00%
    Avg Len  : 6.31 chars

  Generating from RNN + Attention...
    Novelty  : 97.40%
    Diversity: 70.80%
    Avg Len  : 5.25 chars


            QUANTITATIVE EVALUATION REPORT
             Model  Generated  Unique Names  Novelty Rate %  Diversity %  Avg Name Length
       Vanilla RNN        500           455            96.2         91.0             5.79
Bidirectional LSTM        500           415            38.2         83.0             6.31
   RNN + Attention        500           354            97.4         70.8             5.25


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download complete.
